Hyperparams selection

# Libraries

In [18]:
import openpyxl
import pandas as pd
from pathlib import Path
import numpy as np
import sys
from tqdm import tqdm
import optuna
import joblib
import json
from pathlib import Path
import os
from IPython.display import clear_output
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [19]:
# train/test split
sys.path.insert(1, '../utils_functionality/split_utils/')
from split_tools import get_train_test

In [20]:
# hyperparams
sys.path.insert(1, '../utils_functionality/models/')
from modelling2_hyperparams import *

In [21]:
# sklearn models pipeline
sys.path.insert(1, '../utils_functionality/models/')
from modelling2_utils import MLPipeline

In [22]:
RANDOM_STATE = 42

# Datatasets selection

In [23]:
path_data = Path('../data')
path_results = Path('../results/')
path_highlighted = path_results / 'metrics_modelling2_highlight.xlsx'

In [24]:
df_filtered = pd.DataFrame()
for i, page in enumerate(['Net Impact', 'Splashing']):
    df_highlighted = pd.read_excel(path_highlighted, sheet_name=page)
    wb = openpyxl.load_workbook(path_highlighted, data_only=True)
    ws = wb.worksheets[i]

    highlighted_rows = []
    for row in range(1, ws.max_row):
        cell = ws.cell(column=1, row=row)
        bgColor = cell.fill.bgColor.index
        fgColor = cell.fill.fgColor.index
        if bgColor != '00000000':
            highlighted_rows.append(row)
    highlighted_rows = np.array(highlighted_rows) - 2
    df_temp = df_highlighted.iloc[highlighted_rows]
    df_filtered = pd.concat((df_filtered, df_temp))
df_filtered.reset_index(drop=True, inplace=True)

Datasets for parameters selection.

In [25]:
df_filtered

,dataset,target,model,accuracy,f1,precision,recall,roc_auc
0,df_modelling_dimensionless,net_impact,catboostclassifier_net_impact,0.946667,0.900000,0.900000,0.900000,0.931818
1,df_modelling_dimensionless_pf,net_impact,catboostclassifier_net_impact,0.946667,0.900000,0.900000,0.900000,0.931818
2,df_modelling_dimensionless,net_impact,catboostclassifier_smote_net_impact,0.933333,0.878049,0.857143,0.900000,0.922727
3,df_modelling_no_multicollinearity,net_impact,kneighborsclassifier_net_impact_ordenc,0.933333,0.878049,0.857143,0.900000,0.922727
4,df_modelling_no_multicollinearity,net_impact,xgbclassifier_smote_net_impact,0.933333,0.871795,0.894737,0.850000,0.906818
5,df_modelling_no_multicollinearity_pf,net_impact,xgbclassifier_ohe_smote_net_impact,0.933333,0.871795,0.894737,0.850000,0.906818
6,df_modelling_no_multicollinearity_pf,net_impact,kneighborsclassifier_net_impact_onehot,0.920000,0.857143,0.818182,0.900000,0.913636
7,df_modelling_no_multicollinearity,net_impact,xgbclassifier_ohe_smote_net_impact,0.920000,0.850000,0.850000,0.850000,0.897727
8,df_modelling_no_multicollinearity,net_impact,svc_smote_net_impact_onehot,0.893333,0.826087,0.730769,0.950000,0.911364
9,df_modelling_no_multicollinearity_pf,net_impact,logisticregression_net_impact_ordenc,0.906667,0.820513,0.842105,0.800000,0.872727


In [26]:
sorted(df_filtered['model'].unique())

['catboostclassifier_net_impact',
 'catboostclassifier_smote_net_impact',
 'catboostclassifier_smote_splashing',
 'catboostclassifier_splashing',
 'kneighborsclassifier_net_impact_onehot',
 'kneighborsclassifier_net_impact_ordenc',
 'kneighborsclassifier_smote_net_impact_ordenc',
 'kneighborsclassifier_smote_splashing_ordenc',
 'kneighborsclassifier_splashing_ordenc',
 'logisticregression_net_impact_ordenc',
 'logisticregression_splashing_onehot',
 'svc_smote_net_impact_onehot',
 'svc_smote_net_impact_ordenc',
 'svc_smote_splashing_onehot',
 'svc_smote_splashing_ordenc',
 'xgbclassifier_ohe_smote_net_impact',
 'xgbclassifier_ohe_smote_splashing',
 'xgbclassifier_smote_net_impact',
 'xgbclassifier_smote_splashing']

# Parameters selection

In [27]:
metrics = []
f1_global = 0

In [28]:
path_best_model = Path('../results/best_models_modelling_2')
if not os.path.exists(path_best_model): os.makedirs(path_best_model)

In [29]:
def objective(
        trial, train, test, target, model_str, 
        random_state, dataset_filename, cat_features=['wettability']):
    global metrics
    global f1_global
    params = get_params(trial, model_str, random_state, cat_features)
    model = get_model(model_str, params)
    
    preproc_pipeline = get_preproc_pipeline(model_str=model_str, random_state=random_state, 
                                            cat_features=cat_features, 
                                            num_features=dict_num_features[dataset_filename])
    pipeline = [('model', model)]
    if preproc_pipeline: pipeline.insert(0, preproc_pipeline[0])
    pipeline = Pipeline(steps=pipeline)
    pipeline.fit(train.drop(columns=[target]), train[target])
    preds = pipeline.predict(test.drop(columns=[target]))
    f1 = f1_score(test[target], preds)
    if f1 > f1_global:
        metrics = [{
                'accuracy': accuracy_score(test[target], preds),
                'f1': f1,
                'precision': precision_score(test[target], preds),
                'recall': recall_score(test[target], preds),
                'roc_auc': roc_auc_score(test[target], preds)}]
        f1_global = f1
        joblib.dump(pipeline, str(path_best_model / f'{model_str}_{dataset_filename}_{target}.pkl'))
    return f1

In [30]:
path_df_results = '../results/metrics_modelling2.xlsx'
df_results = pd.read_excel(path_df_results)
df_results['optuna_flg'] = 0

In [31]:
df_optuna = pd.DataFrame()
dict_results = {}
for i in (pbar:=tqdm(range(df_filtered.shape[0]))):
    dataset_filename = df_filtered.iloc[i]['dataset']
    if 'df_full' in dataset_filename: continue
    f1_global = 0
    model_str = df_filtered.iloc[i]['model']
    pbar.set_description(f"Processing\t{model_str}\t{dataset_filename}")
    target = df_filtered.iloc[i]['target']
    train, test = get_train_test(
        dataset_filename=dataset_filename,
        target=target)
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize")
    def obj(trial):
        return objective(
            trial, train, test, 
            target, model_str, dataset_filename=dataset_filename, 
            random_state=RANDOM_STATE, cat_features=['wettability'])
    study.optimize(obj, n_trials=600, show_progress_bar=True, n_jobs=-1)
    dict_results[f'{model_str}_{dataset_filename}'] = {
            'dataset': dataset_filename,
            'model': model_str,
            'target': target,
            'metrics': metrics,
            'params': study.best_params}
    df_current = pd.DataFrame({
            'dataset': [dataset_filename],
            'target': [target],
            'model': [f'{model_str}'.replace(f'_{dataset_filename}', '')],
            'optuna_flg': [1],
            'accuracy': [metrics[0]['accuracy']],
            'f1': [metrics[0]['f1']],
            'precision': [metrics[0]['precision']],
            'recall': [metrics[0]['recall']],
            'roc_auc': [metrics[0]['roc_auc']]})
    df_optuna = pd.concat((df_optuna, df_current))
    clear_output()
    metrics = []
df_results = pd.concat((df_results, df_optuna))

Processing	logisticregression_splashing_onehot	df_modelling_dimensionless_pf: 100%|██████████| 32/32 [01:26<00:00,  2.71s/it]


In [32]:
with open("../results/optuna_results.json", "w") as outfile: 
    json.dump(dict_results, outfile)
df_results.drop(columns=['Unnamed: 0'], inplace=True)
df_results.reset_index(drop=True, inplace=True)
df_results.to_excel(path_df_results)